# EMOCA trên Colab cho MA3D-Net Web Demo (ADR-015)

Notebook này dựng EMOCA thật trên Colab, chạy `server.py` (dịch vụ HTTP `emoca_svc`) và mở một tunnel để **web chạy trên máy bạn** gọi tới. Nó thay cho phương án WSL2 cũ (lưu ở `docs/context/archive/PLAN_WSL2_LOCAL.md`).

**Chưa được chạy thử trên Colab bởi người viết** — mọi bước "đo" bên dưới (M0-02, M0-04, M0-05, M1-03) sinh ra số liệu để bạn dán lại cho nhóm; đừng coi là "chạy được" cho tới khi các ô kiểm tra báo PASS.

## Trước khi chạy
1. **Runtime → Change runtime type → GPU.** Cần GPU CUDA.
2. Trên máy bạn: `python services/emoca_svc/pack_for_colab.py` rồi giữ file `notebooks/emoca_svc_bundle.zip` để tải lên ở bước 2.
3. **Giấy phép:** FLAME (flame.is.tue.mpg.de) và EMOCA (emoca.is.tue.mpg.de) cần bạn tự đăng ký và đồng ý điều khoản. Ô "Thông số" có một cờ bạn phải tự đổi sang `True` — notebook không tự đồng ý thay bạn.
4. **Riêng tư:** ảnh bạn gửi từ web sẽ đi qua Google (Colab) và Cloudflare (tunnel). Chỉ dùng ảnh của bạn hoặc ảnh công khai. Máy chủ EMOCA không ghi ảnh ra đĩa.
5. **Token:** notebook sinh một token ngẫu nhiên; ô 9c hiện token MỘT lần rồi xoá khỏi màn hình sau khi bạn bấm Enter. Ai có địa chỉ tunnel mà không có token thì bị từ chối (HTTP 401). **Không chia sẻ notebook sau khi đã chạy** — kết quả các ô chứa token.

## Thứ tự
`0` khởi tạo → `1` thông số → `2` tải mã dự án → `3` dựng môi trường (lâu, ghi lại thời gian) → `4` tài sản (+ tìm ảnh mẫu) → `5` áp patch → `6` smoke test EMOCA (M0-04) → `7` chạy server → `8` kiểm bằng ảnh thật (M1-03, M0-05) → `9` mở tunnel (+ POST thử qua tunnel, hiện token) → `10` giữ phiên → `11` tải kết quả → `12` tắt server và tunnel.

## Giới hạn tái lập (review ADR-015, L7)
Bộ cài Miniconda (`Miniconda3-latest`), `mamba` (cài không kèm số phiên bản) và công cụ tunnel của Cloudflare (bản `latest` trên GitHub) **CHƯA được ghim phiên bản**. Bằng chứng dựng env thành công là từ 2025-11-20 (`emoca_setup_v2_work38_ok.ipynb`); nếu bước 3 hoặc 9 hỏng, hãy ghi lại phiên bản thật đã tải trước khi sửa. Không tự đoán URL/phiên bản để ghim.

## 0. Khởi tạo

In [ ]:
import os, sys, time, json, platform, subprocess, secrets, glob, base64, hashlib, re
import urllib.request, urllib.error

T0 = time.time()
STAGES = []          # (tên bước, số giây) — bằng chứng thời gian dựng thực tế cho M0-02
_last = [T0]

def mark(name):
    now = time.time()
    STAGES.append((name, round(now - _last[0], 1)))
    print(f"[{name}] {now - _last[0]:.0f} s (tổng từ đầu phiên: {now - T0:.0f} s)")
    _last[0] = now

os.makedirs("/content/out", exist_ok=True)
print("Python của kernel:", platform.python_version(), "|", platform.platform())
!nvidia-smi
!df -h /content | tail -1
_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
assert _gpu.returncode == 0 and "GPU" in _gpu.stdout, (
    "Runtime chưa có GPU: Runtime → Change runtime type → Hardware accelerator = T4 GPU, rồi chạy lại từ ô này "
    "(mục 3 dựng env rất lâu — đừng để tới lúc đó mới biết thiếu GPU).")
print("GPU OK:", _gpu.stdout.strip())

## 1. Thông số — sửa ở đây

`ACCEPT_FLAME_EMOCA_LICENSES` mặc định `False`. Chỉ đổi thành `True` khi **chính bạn** đã đăng ký và đồng ý điều khoản của FLAME và EMOCA (script tải tài sản của EMOCA hỏi đúng hai câu đó).

In [ ]:
ACCEPT_FLAME_EMOCA_LICENSES = False      # <-- đổi thành True SAU KHI bạn đã đăng ký + đồng ý điều khoản FLAME và EMOCA
EMOCA_COMMIT = "e0be0db"                 # commit upstream mà bộ patch được viết cho (ADR-001) — đừng đổi nếu chưa kiểm patch
MODEL_NAME = "EMOCA_v2_lr_mse_20"        # mô hình EMOCA v2 khuyến nghị (docs/modules/EMOCA.md §5.3); nhóm xác nhận đây là bản đã dùng cho MA3D-Net (người dùng báo 2026-09-25)
PORT = 8765
# Texture BFM->FLAME (mã EMOCA nạp bắt buộc). Id dưới đây lấy NGUYÊN từ notebook cũ của bạn (Drive cá nhân).
# Kiểm tra quyền sử dụng texture BFM (docs/modules/EMOCA.md §7.4); thay bằng id file của bạn nếu cần.
FLAME_ALBEDO_DRIVE_ID = "1poa4GTzjyArYJ7LhbQN1Gqp2U2zjwiD7"

TOKEN = secrets.token_urlsafe(32)        # sinh mới mỗi phiên; KHÔNG in ở đây, chỉ hiện một lần ở ô 9c
# EX_DIR (thư mục ảnh mẫu công khai của EMOCA) KHÔNG đặt cứng ở đây: ô 4b TÌM nó sau khi tải tài sản
# (review ADR-015 D1: download_assets.sh giải nén vào /content/emoca/data/..., không phải assets/data/... như demo upstream).
print("Đã đặt thông số. Cờ giấy phép:", ACCEPT_FLAME_EMOCA_LICENSES)

## 2. Tải mã của dự án lên

Chọn file `emoca_svc_bundle.zip` (tạo bằng `python services/emoca_svc/pack_for_colab.py` trên máy bạn). Chỉ chứa mã của dự án — **không** có FLAME/checkpoint.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
name = next(n for n in uploaded if n.endswith(".zip"))
os.makedirs("/content/svc", exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall("/content/svc")
manifest = json.load(open("/content/svc/MANIFEST.json"))
bad = [f["path"] for f in manifest["files"]
       if hashlib.sha256(open("/content/svc/" + f["path"], "rb").read()).hexdigest() != f["sha256"]]
assert not bad, f"sha256 không khớp MANIFEST: {bad}"
assert manifest["upstream_commit_expected"] == EMOCA_COMMIT, (
    f"bundle viết cho EMOCA {manifest['upstream_commit_expected']}, notebook đặt {EMOCA_COMMIT}")
assert b"\r" not in open("/content/svc/patches/apply.sh", "rb").read(), "apply.sh còn CRLF — đóng gói lại bằng pack_for_colab.py"
print("Bundle OK:", len(manifest["files"]), "tệp | contract_version", manifest["contract_version"])
mark("tải mã dự án")

## 3. Dựng môi trường `work38` (Python 3.8, torch 1.11/cu115, PyTorch3D 0.6.2)

Các ô bên dưới sao chép **nguyên văn** từ `emoca_setup_v2_work38_ok.ipynb` (đã chạy được), chỉ thêm: ghim commit và đồng hồ đo thời gian. Bước này lâu nhất; thời gian từng chặng nằm ở bảng cuối notebook.

**Những dòng dưới đây CÓ trong lần chạy thành công ngày 2025-11-20 (đọc từ đầu ra đã lưu của `emoca_setup_v2_work38_ok.ipynb`) — bỏ qua được:** `Error downloading object: ibug/face_detection/.../Resnet50_Final.pth` cùng `external filter 'git-lfs filter-process' failed` (submodule `spectre`, không dùng khi suy luận ảnh) và `Terms of Service not found for https://conda.anaconda.org/conda-forge` (chỉ là cảnh báo). Lần đó: mamba 2.3.3, pytorch 1.11.0 + pytorch3d 0.6.2 + cudatoolkit 11.5.1, mmcv-full 1.7.2 (wheel `cu115/torch1.11.0`). Thời gian từng chặng lần đó **không được lưu**, nên mọi con số thời gian trong tài liệu chỉ là ước lượng cho tới khi bảng thời gian cuối notebook có số thật.

**Ô "kiểm nhanh môi trường" ở cuối mục 3 chạy TRƯỚC khi tải tài sản:** nó in phiên bản torch/CUDA/GPU/PyTorch3D và dừng ngay nếu thiếu, để khỏi tốn thời gian tải vài GB rồi mới hỏng.

In [ ]:
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /content/miniconda
!rm miniconda.sh
import os
os.environ['PATH'] = "/content/miniconda/bin:" + os.environ['PATH']
mark('miniconda')

In [ ]:
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/radekd91/emoca.git /content/emoca
%cd /content/emoca
!git checkout {EMOCA_COMMIT}
!git rev-parse --short=7 HEAD
mark('clone + ghim commit')

In [ ]:
yaml_content = '''\
name: work38
channels:
  - pytorch3d
  - pytorch
  - huggingface
  - nvidia
  - defaults
  - anaconda
  - conda-forge
  - https://repo.anaconda.com/pkgs/main
  - https://repo.anaconda.com/pkgs/r
dependencies:
  - _libgcc_mutex=0.1=conda_forge
  - _openmp_mutex=4.5=2_gnu
  - aom=3.3.0=h27087fc_1
  - argon2-cffi=21.3.0=pyhd3eb1b0_0
  - argon2-cffi-bindings=21.2.0=py38h7f8727e_0
  - asttokens=2.0.5=pyhd3eb1b0_0
  - attrs=24.2.0=py38h06a4308_0
  - backcall=0.2.0=pyhd3eb1b0_0
  - beautifulsoup4=4.12.3=py38h06a4308_0
  - blas=1.0=mkl
  - bleach=4.1.0=pyhd3eb1b0_0
  - blosc=1.21.3=h6a678d5_0
  - bottleneck=1.3.7=py38ha9d4c09_0
  - brotli=1.0.9=h5eee18b_9
  - brotli-bin=1.0.9=h5eee18b_9
  - brotli-python=1.0.9=py38h6a678d5_8
  - brunsli=0.1=h2531618_0
  - bzip2=1.0.8=h5eee18b_6
  - c-ares=1.19.1=h5eee18b_0
  - ca-certificates=2024.12.31=h06a4308_0
  - certifi=2024.8.30=py38h06a4308_0
  - cffi=1.17.1=py38h1fdaa30_0
  - cfitsio=3.470=hb418390_7
  - charls=2.2.0=h2531618_0
  - charset-normalizer=2.0.4=pyhd3eb1b0_0
  - click=8.1.7=py38h06a4308_0
  - cloudpickle=3.0.0=py38h06a4308_0
  - comm=0.2.1=py38h06a4308_0
  - contourpy=1.0.5=py38hdb19cb5_0
  - cryptography=43.0.0=py38hdda0065_0
  - cudatoolkit=11.5.1=hcf5317a_9
  - cycler=0.11.0=pyhd3eb1b0_0
  - cyrus-sasl=2.1.28=h52b45da_1
  - cytoolz=0.12.2=py38h5eee18b_0
  - dask-core=2023.4.1=py38h06a4308_0
  - dataclasses=0.8=pyh6d0b6a4_7
  - dbus=1.13.18=hb2f20db_0
  - debugpy=1.6.7=py38h6a678d5_0
  - decorator=4.4.2=pyhd3eb1b0_0
  - defusedxml=0.7.1=pyhd3eb1b0_0
  - executing=0.8.3=pyhd3eb1b0_0
  - expat=2.6.4=h6a678d5_0
  - ffmpeg=4.4.1=habc3f16_5
  - ffmpeg-python=0.2.0=py_0
  - filelock=3.13.1=py38h06a4308_0
  - fontconfig=2.14.1=h55d465d_3
  - fonttools=4.51.0=py38h5eee18b_0
  - freetype=2.12.1=h4a9f257_0
  - freetype-py=2.2.0=pyhd3eb1b0_0
  - fsspec=2024.6.1=py38h06a4308_0
  - future=0.18.3=py38h06a4308_0
  - fvcore=0.1.5.post20221221=pyhd8ed1ab_0
  - giflib=5.2.2=h5eee18b_0
  - glib=2.78.4=h6a678d5_0
  - glib-tools=2.78.4=h6a678d5_0
  - gmp=6.3.0=h6a678d5_0
  - gnutls=3.7.9=hb077bed_0
  - gst-plugins-base=1.14.1=h6a678d5_1
  - gstreamer=1.14.1=h5eee18b_1
  - huggingface_hub=0.28.1=py_0
  - icu=73.1=h6a678d5_0
  - idna=3.7=py38h06a4308_0
  - imagecodecs=2021.8.26=py38hfcb8610_2
  - imageio=2.33.1=py38h06a4308_0
  - imageio-ffmpeg=0.5.1=pyhd8ed1ab_0
  - importlib-metadata=7.0.1=py38h06a4308_0
  - importlib_metadata=7.0.1=hd3eb1b0_0
  - importlib_resources=6.4.0=py38h06a4308_0
  - intel-openmp=2023.1.0=hdb19cb5_46306
  - iopath=0.1.10=pyhd8ed1ab_0
  - ipykernel=6.29.5=py38h06a4308_0
  - ipython=8.12.2=py38h06a4308_0
  - ipython_genutils=0.2.0=pyhd3eb1b0_1
  - jedi=0.19.1=py38h06a4308_0
  - jinja2=3.1.4=py38h06a4308_0
  - joblib=1.4.2=py38h06a4308_0
  - jpeg=9e=h5eee18b_3
  - jsonschema=4.23.0=py38h06a4308_0
  - jsonschema-specifications=2023.7.1=py38h06a4308_0
  - jupyter_client=8.6.0=py38h06a4308_0
  - jupyter_core=5.7.2=py38h06a4308_0
  - jupyterlab_pygments=0.2.2=py38h06a4308_0
  - jxrlib=1.1=h7b6447c_2
  - kiwisolver=1.4.4=py38h6a678d5_0
  - kornia=0.6.5=pyhd8ed1ab_0
  - krb5=1.20.1=h143b758_1
  - lame=3.100=h7b6447c_0
  - lazy_loader=0.4=py38h06a4308_0
  - lcms2=2.16=hb9589c4_0
  - ld_impl_linux-64=2.40=h12ee557_0
  - lerc=3.0=h295c915_0
  - libabseil=20240116.2=cxx17_h6a678d5_0
  - libaec=1.1.3=h6a678d5_0
  - libblas=3.9.0=1_h86c2bf4_netlib
  - libbrotlicommon=1.0.9=h5eee18b_9
  - libbrotlidec=1.0.9=h5eee18b_9
  - libbrotlienc=1.0.9=h5eee18b_9
  - libcblas=3.9.0=5_h92ddd45_netlib
  - libclang=14.0.6=default_hc6dbbc7_2
  - libclang13=14.0.6=default_he11475f_2
  - libcups=2.4.2=h2d74bed_1
  - libcurl=8.11.1=hc9e6f67_0
  - libdeflate=1.17=h5eee18b_1
  - libdrm=2.4.124=hb9d3cd8_0
  - libedit=3.1.20230828=h5eee18b_0
  - libegl=1.7.0=ha4b6fd6_2
  - libev=4.33=h7f8727e_1
  - libexpat=2.6.4=h5888daf_0
  - libffi=3.4.4=h6a678d5_1
  - libgcc=14.2.0=h77fa898_1
  - libgcc-ng=14.2.0=h69a702a_1
  - libgfortran-ng=11.2.0=h00389a5_1
  - libgfortran5=11.2.0=h1234567_1
  - libgl=1.7.0=ha4b6fd6_2
  - libglib=2.78.4=hdc74915_0
  - libglvnd=1.7.0=ha4b6fd6_2
  - libglx=1.7.0=ha4b6fd6_2
  - libgomp=14.2.0=h77fa898_1
  - libiconv=1.17=hd590300_2
  - libidn2=2.3.4=h5eee18b_0
  - liblapack=3.9.0=5_h92ddd45_netlib
  - libllvm11=11.1.0=h9e868ea_6
  - libllvm14=14.0.6=hecde1de_4
  - libnghttp2=1.57.0=h2d74bed_0
  - libpciaccess=0.18=hd590300_0
  - libpng=1.6.39=h5eee18b_0
  - libpq=17.2=hdbd6064_0
  - libprotobuf=4.25.3=he621ea3_0
  - libsodium=1.0.18=h7b6447c_0
  - libssh2=1.11.1=h251f7ec_0
  - libstdcxx=14.2.0=hc0a3c3a_1
  - libstdcxx-ng=14.2.0=h4852527_1
  - libtasn1=4.19.0=h5eee18b_0
  - libtiff=4.5.1=h6a678d5_0
  - libunistring=0.9.10=h27cfd23_0
  - libuuid=1.41.5=h5eee18b_0
  - libuv=1.48.0=h5eee18b_0
  - libva=2.22.0=h8a09558_1
  - libvpx=1.11.0=h295c915_0
  - libwebp-base=1.3.2=h5eee18b_1
  - libxcb=1.17.0=h8a09558_0
  - libxkbcommon=1.0.1=h097e994_2
  - libxml2=2.13.5=hfdd30dd_0
  - libzlib=1.2.13=h4ab18f5_6
  - libzopfli=1.0.3=he6710b0_0
  - llvmlite=0.38.0=py38h4ff587b_0
  - locket=1.0.0=py38h06a4308_0
  - lz4-c=1.9.4=h6a678d5_1
  - markupsafe=2.1.3=py38h5eee18b_0
  - matplotlib=3.7.2=py38h06a4308_0
  - matplotlib-base=3.7.2=py38h1128e8f_0
  - matplotlib-inline=0.1.6=py38h06a4308_0
  - mistune=2.0.4=py38h06a4308_0
  - mkl=2023.1.0=h213fc3f_46344
  - mkl-service=2.4.0=py38h5eee18b_1
  - mkl_fft=1.3.8=py38h5eee18b_0
  - mkl_random=1.2.4=py38hdb19cb5_0
  - moviepy=1.0.1=py_0
  - multidict=5.1.0=py38h27cfd23_2
  - murmurhash=1.0.5=py38h2531618_0
  - mysql=8.4.0=h29a9f33_1
  - nbclient=0.8.0=py38h06a4308_0
  - nbconvert=7.16.4=py38h06a4308_0
  - nbformat=5.10.4=py38h06a4308_0
  - ncurses=6.4=h6a678d5_0
  - nest-asyncio=1.6.0=py38h06a4308_0
  - nettle=3.9.1=h7ab15ed_0
  - networkx=3.1=py38h06a4308_0
  - ninja=1.10.2=h06a4308_5
  - ninja-base=1.10.2=hd09550d_5
  - notebook=6.3.0=py38h06a4308_0
  - numba=0.55.1=py38h51133e4_0
  - numexpr=2.8.4=py38hc78ab66_1
  - numpy=1.21.6=py38h5f9d8c6_1
  - numpy-base=1.21.6=py38hb5e798b_1
  - openh264=2.1.1=h4ff587b_0
  - openjpeg=2.5.2=he7f1fd0_0
  - openldap=2.6.4=h42fbc30_0
  - openssl=3.0.15=h5eee18b_0
  - p11-kit=0.24.1=hc5aa10d_0
  - packaging=24.1=py38h06a4308_0
  - pandas=2.0.3=py38h1128e8f_0
  - pandocfilters=1.5.0=pyhd3eb1b0_0
  - parso=0.8.3=pyhd3eb1b0_0
  - partd=1.4.1=py38h06a4308_0
  - pcre2=10.42=hebb0a14_1
  - pexpect=4.8.0=pyhd3eb1b0_3
  - pickleshare=0.7.5=pyhd3eb1b0_1003
  - pillow=10.4.0=py38h5eee18b_0
  - pip=24.2=py38h06a4308_0
  - pkgutil-resolve-name=1.3.10=py38h06a4308_1
  - platformdirs=3.10.0=py38h06a4308_0
  - ply=3.11=py38_0
  - portalocker=2.3.0=py38h06a4308_1
  - proglog=0.1.10=pyhaa61c55_0
  - prometheus_client=0.14.1=py38h06a4308_0
  - prompt-toolkit=3.0.43=py38h06a4308_0
  - protobuf=4.25.3=py38h12ddb61_0
  - psutil=5.9.0=py38h5eee18b_0
  - pthread-stubs=0.3=h0ce48e5_1
  - ptyprocess=0.7.0=pyhd3eb1b0_2
  - pure_eval=0.2.2=pyhd3eb1b0_0
  - pycparser=2.21=pyhd3eb1b0_0
  - pyglet=1.5.27=py38h578d9bd_4
  - pygments=2.15.1=py38h06a4308_1
  - pyopengl=3.1.1a1=py38h06a4308_0
  - pyopenssl=24.2.1=py38h06a4308_0
  - pyparsing=3.0.9=py38h06a4308_0
  - pyqt=5.15.10=py38h6a678d5_0
  - pyqt5-sip=12.13.0=py38h5eee18b_0
  - pyrender=0.1.45=pyh8a188c0_3
  - pysocks=1.7.1=py38h06a4308_0
  - python=3.8.20=he870216_0
  - python-dateutil=2.9.0post0=py38h06a4308_2
  - python-fastjsonschema=2.16.2=py38h06a4308_0
  - python-tzdata=2023.3=pyhd3eb1b0_0
  - python_abi=3.8=1_cp38
  - pytorch=1.11.0=py3.8_cuda11.5_cudnn8.3.2_0
  - pytorch-mutex=1.0=cuda
  - pytorch3d=0.6.2=py38_cu115_pyt1110
  - pytz=2024.1=py38h06a4308_0
  - pywavelets=1.4.1=py38h5eee18b_0
  - pyyaml=6.0.2=py38h5eee18b_0
  - pyzmq=25.1.2=py38h6a678d5_0
  - qt-main=5.15.2=hb6262e9_11
  - readline=8.2=h5eee18b_0
  - referencing=0.30.2=py38h06a4308_0
  - regex=2024.9.11=py38h5eee18b_0
  - requests=2.27.1=pyhd3eb1b0_0
  - resampy=0.2.2=py_0
  - rpds-py=0.10.6=py38hb02cf49_0
  - sacremoses=master=py_0
  - scikit-image=0.20.0=py38h6a678d5_0
  - scikit-learn=1.3.0=py38h1128e8f_1
  - scikit-video=1.1.11=pyh24bf2e0_0
  - scipy=1.9.1=py38hea3f02b_0
  - seaborn=0.11.0=py_0
  - send2trash=1.8.2=py38h06a4308_0
  - setuptools=75.1.0=py38h06a4308_0
  - sip=6.7.12=py38h6a678d5_0
  - six=1.16.0=pyhd3eb1b0_1
  - snappy=1.2.1=h6a678d5_0
  - soupsieve=2.5=py38h06a4308_0
  - sqlite=3.45.3=h5eee18b_0
  - stack_data=0.2.0=pyhd3eb1b0_0
  - svt-av1=1.1.0=h27087fc_1
  - tabulate=0.9.0=py38h06a4308_0
  - tbb=2021.8.0=hdb19cb5_0
  - termcolor=2.1.0=py38h06a4308_0
  - terminado=0.17.1=py38h06a4308_0
  - threadpoolctl=3.5.0=py38h2f386ee_0
  - tifffile=2021.7.2=pyhd3eb1b0_2
  - tinycss2=1.2.1=py38h06a4308_0
  - tk=8.6.14=h39e8969_0
  - tokenizers=0.10.3=py38_0
  - tomli=2.0.1=py38h06a4308_0
  - toolz=0.12.0=py38h06a4308_0
  - torchaudio=0.11.0=py38_cu115
  - torchvision=0.12.0=py38_cu115
  - tornado=6.4.1=py38h5eee18b_0
  - tqdm=4.66.5=py38h2f386ee_0
  - traitlets=5.14.3=py38h06a4308_0
  - transformers=4.11.3=py_0
  - trimesh=4.5.3=pyhd8ed1ab_0
  - typing-extensions=4.11.0=py38h06a4308_0
  - typing_extensions=4.11.0=py38h06a4308_0
  - unicodedata2=15.1.0=py38h5eee18b_0
  - urllib3=1.26.19=py38h06a4308_0
  - wayland=1.23.1=h3e06ad9_0
  - wayland-protocols=1.40=hd8ed1ab_0
  - wcwidth=0.2.5=pyhd3eb1b0_0
  - webencodings=0.5.1=py38_1
  - wheel=0.44.0=py38h06a4308_0
  - x264=1!161.3030=h7f98852_1
  - x265=3.5=h924138e_3
  - xorg-libx11=1.8.11=h4f16b4b_0
  - xorg-libxau=1.0.12=hb9d3cd8_0
  - xorg-libxdmcp=1.1.5=hb9d3cd8_0
  - xorg-libxext=1.3.6=hb9d3cd8_0
  - xorg-libxfixes=6.0.1=hb9d3cd8_0
  - xz=5.4.6=h5eee18b_1
  - yacs=0.1.6=pyhd3eb1b0_1
  - yaml=0.2.5=h7b6447c_0
  - yarl=1.11.0=py38h5eee18b_0
  - zeromq=4.3.5=h6a678d5_0
  - zfp=0.5.5=h295c915_6
  - zipp=3.20.2=py38h06a4308_0
  - zlib=1.2.13=h4ab18f5_6
  - zstd=1.5.6=hc292b87_0
prefix: /content/miniconda/envs/work38
'''
with open("environment.yml", "w") as file:
    file.write(yaml_content)

print("YAML file saved as environment.yml")
mark('environment.yml')

In [ ]:
!conda config --set channel_priority strict
!conda config --add channels defaults
!conda config --add channels conda-forge

# Accept TOS for required channels
!conda tos accept --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --channel https://repo.anaconda.com/pkgs/r

In [ ]:
!echo "Pulling submodules"
!bash pull_submodules.sh
!echo "Installing mamba"
!conda install mamba -n base -c conda-forge -y
!if ! command -v mamba &> /dev/null; \
    then echo "mamba could not be found. Please install mamba before running this script"; \
    exit; \
fi

In [ ]:
!mamba create -n work38 python=3.8 -y

!conda info --envs

!conda config --set channel_priority flexible

In [ ]:
!source activate work38; \
    mamba env list; \
    mamba env update -n work38 --file environment.yml --prune -y
mark('mamba env update (nặng nhất)')

In [ ]:
!source activate work38; \
    conda list torch; \
    conda list cudatoolkit

!source activate work38; \
    pip install --upgrade pip==24.0

In [ ]:
with open("/content/emoca/requirements38.txt", "r") as file:
    lines = file.readlines()

for i, line in enumerate(lines):
    if line.startswith("onnx"): # 2 onnx
        lines[i] = "# " + line
    if line.startswith("youtube-dl"):
        lines[i] = "# " + line
    if line.startswith("mediapipe"):
        lines[i] = "# " + line
    if line.startswith("mmcv-full"):
        lines[i] = "# " + line
    if line.startswith("pyzmq"):
        lines[i] = "# " + line
    # if line.startswith("numpy"):
    #     lines[i] = "# " + line

with open("/content/emoca/requirements38.txt", "w") as file:
    file.writelines(lines)

In [ ]:
!source activate work38; \
    pip install -r requirements38.txt; \
    pip install Cython==0.29;

In [ ]:
!source activate work38; \
    pip install onnx==1.10.2 onnxruntime-gpu youtube-dl~=2021.6.6 mediapipe pyzmq==26.2.1;

!source activate work38; \
    pip install mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu115/torch1.11/index.html;

In [ ]:
!source activate work38; \
    echo "Installing GDL"; \
    pip install -e . ; \
    echo "Installation finished"
mark('pip + pip install -e .')

In [ ]:
# Kiểm nhanh môi trường NGAY (trước khi tải tài sản nặng): thiếu GPU / torch / pytorch3d thì dừng ở đây.
_probe = ("import torch, pytorch3d, face_alignment; "
          "assert torch.cuda.is_available(), 'torch không thấy GPU'; "
          "print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0)); "
          "print('pytorch3d', pytorch3d.__version__, '| face_alignment', face_alignment.__version__)")
_r = subprocess.run(["bash", "-c", 'source activate work38 && python -c "' + _probe + '"'], capture_output=True, text=True)
print(_r.stdout.strip())
assert _r.returncode == 0, "Môi trường work38 chưa dùng được — gửi phần dưới đây cho nhóm:" + "\n" + _r.stderr[-1500:]
mark('kiểm nhanh môi trường')

## 4. Tài sản có giấy phép (FLAME, EMOCA, texture BFM)

Ô đầu **dừng** nếu bạn chưa đổi cờ `ACCEPT_FLAME_EMOCA_LICENSES`. Script `download_assets.sh` của EMOCA hỏi hai câu xác nhận giấy phép; ở đây trả lời `y` tự động **chỉ vì bạn đã tự khẳng định ở ô Thông số**.

In [ ]:
assert ACCEPT_FLAME_EMOCA_LICENSES, (
    "Chưa đồng ý giấy phép: đăng ký ở flame.is.tue.mpg.de và emoca.is.tue.mpg.de, đọc điều khoản, "
    "rồi đặt ACCEPT_FLAME_EMOCA_LICENSES = True ở ô Thông số.")
%cd /content/emoca/gdl_apps/EMOCA/demos
!source activate work38; yes | bash download_assets.sh
mark('download_assets.sh (FLAME + EMOCA + ảnh mẫu)')

In [ ]:
# 4b. TÌM ảnh mẫu công khai do download_assets.sh vừa tải (review ADR-015 D1). Không đoán tên thư mục: chỉ dùng glob.
patterns = ["/content/emoca/**/EMOCA_test_example_data/images/**/*.jpg",
            "/content/emoca/**/EMOCA_test_example_data/images/**/*.png"]
found = sorted({p for pat in patterns for p in glob.glob(pat, recursive=True)})
assert found, ("Không thấy ảnh mẫu nào khớp EMOCA_test_example_data/images/** dưới /content/emoca. Xem lại đầu ra ô tải "
               "tài sản ở trên: cuối download_assets.sh phải có 'Downloading example test data' và unzip "
               "EMOCA_test_example_data.zip thành công.")
EX_DIR = os.path.dirname(found[0])
n_dirs = len({os.path.dirname(p) for p in found})
n_here = sum(os.path.dirname(p) == EX_DIR for p in found)
print(f"Tìm thấy {len(found)} ảnh mẫu trong {n_dirs} thư mục. Dùng EX_DIR = {EX_DIR} ({n_here} ảnh).")


In [ ]:
!source activate work38; \
    pip uninstall opencv-python -y; \
    pip uninstall opencv-contrib-python -y; \
    pip uninstall opencv-contrib-python-headless ; \
    pip install opencv-python-headless==4.5.5.64;

!source activate work38; \
    pip install git+https://github.com/mattloper/chumpy

!source activate work38; \
    pip install tensorboard==2.9.1

In [ ]:
import os
os.makedirs('/content/emoca/assets/FLAME/texture', exist_ok=True)
TEX = '/content/emoca/assets/FLAME/texture/FLAME_albedo_from_BFM.npz'
!gdown {FLAME_ALBEDO_DRIVE_ID} --output {TEX}
assert os.path.isfile(TEX) and os.path.getsize(TEX) > 1_000_000, (
    "Tải texture thất bại (Drive hết lượt tải hoặc file không còn chia sẻ công khai). Cách khác: tải "
    "FLAME_albedo_from_BFM.npz về máy, chạy `from google.colab import files; files.upload()`, rồi "
    f"`!mv FLAME_albedo_from_BFM.npz {TEX}` và chạy lại ô này từ dòng assert.")
mark('texture BFM->FLAME + opencv/chumpy/tensorboard')

## 5. Áp bộ patch của dự án (không sửa tại chỗ — CLAUDE.md §1.3)

`0001` (`np.int` → `np.int32`) và `0002` (`face_alignment` đổi tên `LandmarksType`). Bản vá `FaceVideoDataModule` (tuỳ chọn) chỉ cần cho demo video nên **không** áp. Script kiểm đúng commit `e0be0db` và idempotent.

In [ ]:
!bash /content/svc/patches/apply.sh /content/emoca
!git -C /content/emoca status --short
!git -C /content/emoca diff --stat | tail -3
mark('áp patch')

In [ ]:
# Bằng chứng tái lập môi trường (M0-02): phiên bản thật đã dùng.
!source activate work38; conda env export > /content/out/env_work38.yml
!source activate work38; pip freeze > /content/out/pip_freeze_work38.txt
!source activate work38; python -c "import torch, pytorch3d, face_alignment, numpy; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0)); print('pytorch3d', pytorch3d.__version__, '| face_alignment', face_alignment.__version__, '| numpy', numpy.__version__)"
!git -C /content/emoca rev-parse HEAD > /content/out/emoca_commit.txt; cat /content/out/emoca_commit.txt
!du -sh /content/miniconda/envs/work38 /content/emoca/assets 2>/dev/null   # dung lượng thật — cần để quyết định lưu cache lên Drive

## 6. Smoke test EMOCA bằng chính script upstream (M0-04)

Chạy `test_emoca_on_images.py` trên ảnh mẫu **công khai** đi kèm EMOCA. AC của M0-04: có `inputs`, `geometry_coarse`, `geometry_detail`, `out_im_coarse`, `out_im_detail` (.png) và `shape/exp/tex/pose/detail.npy` dài 100/50/50/6/128. (Lưu ý: cờ `--save_codes` ở script này khai báo `type=bool`, nên `False` cũng thành `True` — đó là lỗi upstream đã ghi ở `docs/modules/EMOCA.md`.)

In [ ]:
%cd /content/emoca/gdl_apps/EMOCA/demos
!source activate work38; python test_emoca_on_images.py --input_folder {EX_DIR} --output_folder /content/out/m0_04 --model_name {MODEL_NAME} --save_codes True --save_mesh True 2>&1 | tee /content/out/m0_04_log.txt
# TestData in 'please check the test path' rồi exit() với mã 0 khi thư mục ảnh sai ⇒ lệnh trên trông như chạy xong.
_log = open("/content/out/m0_04_log.txt", encoding="utf-8", errors="replace").read()
if "please check the test path" in _log:
    raise RuntimeError(f"LỖI: test_emoca_on_images.py không thấy thư mục ảnh {EX_DIR!r} (TestData thoát mã 0 nên "
                       "không tự báo lỗi). Chạy lại ô 4b tìm ảnh mẫu.")
mark('M0-04: EMOCA trên ảnh mẫu')

In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display

root = f"/content/out/m0_04/{MODEL_NAME}"
want_png = ["inputs", "geometry_coarse", "geometry_detail", "out_im_coarse", "out_im_detail"]
want_npy = {"shape": 100, "exp": 50, "tex": 50, "pose": 6, "detail": 128}
report, shown = [], 0
for d in sorted(glob.glob(root + "/*")):
    miss_png = [n for n in want_png if not os.path.isfile(f"{d}/{n}.png")]
    bad_npy = {n: (None if not os.path.isfile(f"{d}/{n}.npy") else int(np.load(f"{d}/{n}.npy").reshape(-1).size))
               for n in want_npy}
    bad_npy = {n: v for n, v in bad_npy.items() if v != want_npy[n]}
    report.append((os.path.basename(d), not miss_png and not bad_npy, miss_png, bad_npy))
    if shown < 2 and not miss_png:
        shown += 1
        tiles = [Image.open(f"{d}/{n}.png").convert("RGB").resize((224, 224)) for n in want_png]
        strip = Image.new("RGB", (224 * len(tiles), 224))
        for i, t in enumerate(tiles):
            strip.paste(t, (224 * i, 0))
        print(os.path.basename(d), "→ inputs | geometry_coarse | geometry_detail | out_im_coarse | out_im_detail")
        strip.save(f"/content/out/m0_04_strip_{shown}.png")   # lưu để gửi cho nhóm (tiêu chí (4) cần mắt người)
        display(strip)
ok = [r for r in report if r[1]]
print(f"M0-04: {len(ok)}/{len(report)} ảnh đủ 5 PNG + 5 NPY đúng độ dài")
for r in report:
    if not r[1]:
        print("  THIẾU/SAI:", r)
assert report and len(ok) == len(report), "M0-04 chưa đạt — xem các dòng THIẾU/SAI ở trên"
print("→ M0-04 (1)-(3) ĐẠT. Tiêu chí (4) 'ảnh dựng nhìn đúng khuôn mặt' cần MẮT NGƯỜI: xem dải ảnh phía trên.")

## 7. Chạy `server.py` (dịch vụ EMOCA thật)

Nạp mô hình **một lần** rồi phục vụ. Chỉ nghe `127.0.0.1` — tunnel (bước 9) nối vào đây. Đòi `Authorization: Bearer <token>`.

In [ ]:
def call(base, path, token, body=None, timeout=120):
    headers = {"Authorization": "Bearer " + token, "User-Agent": "emoca-colab-notebook/1.0"}
    if body is not None:
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(base + path, data=body, method="POST" if body is not None else "GET", headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            status, raw = r.status, r.read()
    except urllib.error.HTTPError as e:
        status, raw = e.code, e.read()
    try:
        return status, json.loads(raw)
    except ValueError:
        return status, {"non_json": raw[:200].decode("utf-8", "replace")}

LOCAL = f"http://127.0.0.1:{PORT}"
cmd = ("source /content/miniconda/bin/activate work38 && cd /content/svc && python -u server.py "
       f"--config /content/svc/configs/services.yaml --models-dir /content/emoca/assets/EMOCA/models "
       f"--model-name {MODEL_NAME} --port {PORT}")
server_log = open("/content/server.log", "w")
server = subprocess.Popen(["bash", "-c", cmd], stdout=server_log, stderr=subprocess.STDOUT,
                          env=dict(os.environ, EMOCA_TOKEN=TOKEN, PYTHONUNBUFFERED="1"))
t_wait = time.time()
health = None
while time.time() - t_wait < 900:
    if server.poll() is not None:
        print(open("/content/server.log").read()[-3000:])
        raise RuntimeError("server.py đã thoát — xem nhật ký ở trên")
    try:
        st, body = call(LOCAL, "/v1/health", TOKEN, timeout=5)
        if st == 200:
            health = body
            break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)
assert health, "server không lên sau 15 phút — xem /content/server.log"
mark('server nạp mô hình + sẵn sàng')
print(json.dumps(health, ensure_ascii=False, indent=2))

## 8. Kiểm bằng ảnh thật qua HTTP (M1-03) và hệ toạ độ landmark (M0-05)

Gọi `POST /v1/analyze` trên vài ảnh mẫu công khai, mỗi ảnh vài lần: kiểm đúng schema, hai lần gọi liên tiếp **không nạp lại mô hình** (M1-03 AC2), đo độ trễ và VRAM (AC3), không thấy mặt → `no_face` (AC5). Số liệu in ra là thứ cần dán lại cho nhóm.

In [ ]:
import jsonschema
schema = json.load(open("/content/svc/contracts/emoca_analyze.schema.json"))
imgs = sorted(glob.glob(EX_DIR + "/*.jpg") + glob.glob(EX_DIR + "/*.png"))[:3]
assert imgs, f"không thấy ảnh mẫu trong {EX_DIR}"
load_before = health["load_seconds"]

def analyze_file(path, max_faces=1):
    data = open(path, "rb").read()
    payload = json.dumps({"image_b64": base64.b64encode(data).decode("ascii"), "options": {"max_faces": max_faces}}).encode()
    t = time.perf_counter()
    status, body = call(LOCAL, "/v1/analyze", TOKEN, payload)
    return status, body, (time.perf_counter() - t) * 1000

rows, first_ok = [], None
for p in imgs:
    for k in range(4):
        st, body, ms = analyze_file(p)
        assert st == 200, f"{os.path.basename(p)}: HTTP {st} {body}"
        jsonschema.validate(body, schema)
        f = body["faces"][0]
        rows.append({"ảnh": os.path.basename(p), "lần": k + 1, "khứ_hồi_ms": round(ms), **{k2: v for k2, v in body["timing_ms"].items()}})
        if first_ok is None:
            first_ok = (p, body)
        assert {k2: len(v) for k2, v in f["codes"].items()} == {"shape": 100, "exp": 50, "tex": 50, "pose": 6, "detail": 128}
        assert len(f["landmarks"]["crop224"]) == 68 and len(f["landmarks"]["original_px"]) == 68
print("PASS: mọi phản hồi đúng emoca_analyze.schema.json, codes 100/50/50/6/128, 68 landmark ở hai hệ toạ độ")
print(f"{'ảnh':<28}{'lần':>4}{'khứ hồi':>9}{'detect':>9}{'encode':>9}{'decode':>9}{'total':>9}  (ms)")
for r in rows:
    print(f"{r['ảnh']:<28}{r['lần']:>4}{r['khứ_hồi_ms']:>9}{r['detect']:>9.0f}{r['encode']:>9.0f}{r['decode']:>9.0f}{r['total']:>9.0f}")
warm = sorted(r["total"] for r in rows if r["lần"] >= 2)
print(f"\nĐộ trễ phía máy chủ sau khởi động nóng (bỏ lần 1 của mỗi ảnh, n={len(warm)}): "
      f"p50 = {warm[len(warm) // 2]:.0f} ms, max = {warm[-1]:.0f} ms")
_, health2 = call(LOCAL, "/v1/health", TOKEN)
assert health2["load_seconds"] == load_before, "mô hình bị nạp lại giữa các lần gọi!"
print("M1-03 AC2 ĐẠT: load_seconds không đổi (", load_before, "s) sau", health2["requests_served"], "lần gọi — không nạp lại mô hình")
print("VRAM: sau nạp", health2.get("vram_after_load_mb"), "MB | đỉnh", health2.get("vram_peak_mb"), "MB | GPU", health2.get("device"))
print("Miền landmark chuẩn hoá đã thấy (M0-05, dự kiến nằm trong [-1, 1]):", health2["landmark_norm_range_seen"])
print("Cảnh báo của mặt đầu:", first_ok[1]["faces"][0]["warnings"])
json.dump(first_ok[1], open("/content/out/analyze_response_sample.json", "w"), ensure_ascii=False)
json.dump({"stages_s": STAGES, "rows": rows, "health": health2}, open("/content/out/measurements.json", "w"), ensure_ascii=False, indent=1)

# M1-03 AC5: ảnh không có mặt → no_face (không rơi về "chạy trên ảnh nguyên")
from PIL import Image as _I
import io as _io
blank = _io.BytesIO(); _I.new("RGB", (320, 320), (128, 128, 128)).save(blank, "PNG")
payload = json.dumps({"image_b64": base64.b64encode(blank.getvalue()).decode("ascii")}).encode()
st, body = call(LOCAL, "/v1/analyze", TOKEN, payload)
print("Ảnh xám không có mặt →", st, body.get("error"))
assert st == 422 and body["error"]["code"] == "no_face", "M1-03 AC5 chưa đạt"
print("M1-03 AC5 ĐẠT: không thấy mặt → no_face 422")

In [ ]:
%%writefile /content/svc/m0_05_compare.py
# M0-05: so landmark EMOCA (đã đưa về original_px bằng crop_transform của server) với landmark FAN (cũng original_px).
# Hai bộ cùng quy ước 68 điểm iBUG, nên NME theo khoảng cách hai khoé mắt ngoài là một phép thử "có đúng hệ toạ độ không".
# Đây KHÔNG phải độ chính xác của EMOCA (không có nhãn thật) — chỉ bắt lỗi kiểu sai toạ độ/sai phép biến đổi.
import json, sys
import numpy as np
from PIL import Image, ImageDraw
from gdl.utils.FaceDetector import FAN

img_path, resp_path, out_png = sys.argv[1:4]
img = np.array(Image.open(img_path).convert("RGB"))
resp = json.load(open(resp_path))
face = resp["faces"][0]
emoca = np.array(face["landmarks"]["original_px"], dtype=float)
boxes, _, kpts = FAN().run(img, with_landmarks=True)
area = [(b[2] - b[0]) * (b[3] - b[1]) for b in boxes]
fan = np.array(kpts[int(np.argmax(area))], dtype=float).reshape(68, 2)
iod = np.linalg.norm(fan[36] - fan[45])
d = np.linalg.norm(emoca - fan, axis=1) / iod
regions = {"hàm 0-16": range(0, 17), "lông mày 17-26": range(17, 27), "mũi 27-35": range(27, 36),
           "mắt 36-47": range(36, 48), "miệng 48-67": range(48, 68)}
print(f"khoảng cách hai khoé mắt ngoài (FAN): {iod:.1f} px")
print(f"NME toàn bộ: mean {d.mean():.3f} | median {np.median(d):.3f} | max {d.max():.3f}  (đơn vị: khoảng cách hai khoé mắt)")
for name, idx in regions.items():
    print(f"  {name:<16} mean {d[list(idx)].mean():.3f}")
print("Gợi ý đọc (KHÔNG phải ngưỡng nghiệm thu): giá trị nhỏ (vài phần trăm) → hai bộ landmark chồng khít; "
      "lớn hơn ~0.3 hoặc lệch một chiều → nghi sai hệ toạ độ/phép biến đổi.")
out = Image.fromarray(img).convert("RGB")
dr = ImageDraw.Draw(out)
r = max(2, int(max(out.size) * 0.004))
for (x, y) in fan:
    dr.ellipse([x - r, y - r, x + r, y + r], outline=(0, 255, 0))
for (x, y) in emoca:
    dr.ellipse([x - r, y - r, x + r, y + r], fill=(255, 0, 0))
x0, y0, x1, y1 = face["bbox_original_px"]
dr.rectangle([x0, y0, x1, y1], outline=(255, 255, 0), width=max(1, r // 2))
out.save(out_png)
print("Đã lưu ảnh chồng:", out_png, "(đỏ = EMOCA đã đổi về original_px, viền xanh = FAN, khung vàng = bbox_original_px)")

In [ ]:
sample_path, sample_body = first_ok
json.dump(sample_body, open("/content/out/_resp_for_m0_05.json", "w"))
!source activate work38; python /content/svc/m0_05_compare.py "{sample_path}" /content/out/_resp_for_m0_05.json /content/out/m0_05_overlay.png
from IPython.display import Image as _Img, display as _display
_display(_Img("/content/out/m0_05_overlay.png", width=520))
print("→ M0-05 (3): NHÌN ảnh: chấm đỏ phải rơi đúng mắt/mũi/miệng/đường hàm trên ẢNH GỐC. Chụp lại ảnh này cho nhóm (docs/qa/M0/).")

## 9. Mở tunnel (Cloudflare quick tunnel, không cần tài khoản)

Địa chỉ `https://….trycloudflare.com` **đổi mỗi phiên**. Web trên máy bạn dán địa chỉ + token vào bảng **Cài đặt** ở đầu trang (mục *Kết nối EMOCA*), hoặc đặt biến môi trường `EMOCA_MODE=real`, `EMOCA_URL`, `EMOCA_TOKEN`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        URL = m.group(0)
        break
    if tunnel.poll() is not None:
        break
    time.sleep(1)
assert URL, "không lấy được địa chỉ tunnel:\n" + open("/content/tunnel.log").read()[-1500:]

# Địa chỉ mới cần vài giây để DNS lan ra: thử lại cho tới khi qua tunnel gọi được /v1/health
ok = False
for _ in range(45):
    try:
        st, body = call(URL, "/v1/health", TOKEN, timeout=10)
        if st == 200:
            ok = True
            break
        print("chờ tunnel…", st)
    except (urllib.error.URLError, ConnectionError, OSError) as e:
        print("chờ tunnel…", type(e).__name__)
    time.sleep(2)
assert ok, "tunnel chưa nối được tới server sau ~90 giây — xem /content/tunnel.log"
st_bad, _ = call(URL, "/v1/health", "sai-token", timeout=10)
assert st_bad == 401, f"token sai phải bị 401, được {st_bad}"
mark('tunnel')
print("Tunnel hoạt động; token sai bị từ chối 401 (đúng).")

In [ ]:
# 9b. POST THỬ một ảnh mẫu công khai QUA TUNNEL (review ADR-015 D2). Mục 8 chỉ gọi 127.0.0.1; đây là đường web dùng thật:
# nếu proxy/tunnel đổi thân request (vd chuyển sang chunked, cắt thân lớn) thì lộ ra ở đây, trước khi dán URL vào web.
sample = imgs[0]
data = open(sample, "rb").read()
payload = json.dumps({"image_b64": base64.b64encode(data).decode("ascii"), "options": {"max_faces": 1}}).encode()
t = time.perf_counter()
st, body = call(URL, "/v1/analyze", TOKEN, payload, timeout=120)
roundtrip_ms = (time.perf_counter() - t) * 1000
print(f"POST /v1/analyze qua tunnel: HTTP {st}, khứ hồi {roundtrip_ms:.0f} ms, ảnh {os.path.basename(sample)} ({len(data)} byte)")
assert st == 200, f"POST qua tunnel hỏng: HTTP {st} {str(body)[:300]} — xem /content/server.log và /content/tunnel.log"
jsonschema.validate(body, schema)
assert {k: len(v) for k, v in body["faces"][0]["codes"].items()} == {"shape": 100, "exp": 50, "tex": 50, "pose": 6, "detail": 128}
print("PASS: phản hồi qua tunnel đúng emoca_analyze.schema.json · timing_ms phía máy chủ:", body["timing_ms"])
json.dump({"http_status": st, "roundtrip_ms": round(roundtrip_ms), "image_bytes": len(data), "server_timing_ms": body["timing_ms"]},
          open("/content/out/tunnel_post.json", "w"), ensure_ascii=False, indent=1)
mark('POST thử qua tunnel')


In [ ]:
print("=" * 70)
print("DÁN VÀO WEB (⚙ Cài đặt ở đầu trang → Kết nối EMOCA → Máy chủ EMOCA):")
print("  Địa chỉ:", URL)
print("  Token  :", TOKEN[:4] + "…", " ← chạy ô 9c ngay dưới để hiện token đầy đủ MỘT lần")
print("Hoặc đặt biến môi trường trước khi chạy web:")
print(f'  EMOCA_MODE=real  EMOCA_URL={URL}  EMOCA_TOKEN=<token hiện ở ô 9c>')
print("=" * 70)
print("Nhắc lại: ảnh đi qua Google (Colab) + Cloudflare. Chỉ dùng ảnh của bạn hoặc ảnh công khai.")
print("Bảng thời gian dựng (M0-02):")
for n, s in STAGES:
    print(f"  {s:>7.1f} s  {n}")
print(f"  {sum(s for _, s in STAGES):>7.1f} s  TỔNG")

In [ ]:
# 9c. Hiện token một lần, rồi XOÁ khỏi đầu ra của ô này sau khi bạn đã sao chép (đầu ra ô được lưu cùng notebook trên Drive).
from IPython.display import clear_output
print("Token:", TOKEN)
input("Sao chép token vào web rồi nhấn Enter để xoá nó khỏi màn hình… ")
clear_output()
print("Đã xoá token khỏi đầu ra ô này (vẫn còn trong biến TOKEN của phiên; chạy lại ô này nếu cần xem lại).")


## 10. Giữ phiên

Chạy ô này để phiên không bị coi là nhàn rỗi. Nó in một dòng mỗi 5 phút và dừng khi bạn bấm ■ (Interrupt). Colab vẫn có thể ngắt phiên bất cứ lúc nào (giới hạn thời gian/GPU của Colab) — khi đó web sẽ báo `service_unavailable`/`timeout` và cần chạy lại notebook với địa chỉ + token mới.

In [ ]:
print("Đang giữ phiên. Bấm ■ để dừng.")
try:
    n = 0
    while True:
        time.sleep(60)
        n += 1
        if n % 5 == 0:
            alive = (server.poll() is None) and (tunnel.poll() is None)
            try:
                st, h = call(URL, "/v1/health", TOKEN, timeout=20)
            except Exception as e:
                st, h = type(e).__name__, {}
            print(time.strftime("%H:%M:%S"), "tiến trình sống:", alive, "| health:", st, "| đã phục vụ:", h.get("requests_served"),
                  "| VRAM đỉnh MB:", h.get("vram_peak_mb"))
            if not alive:
                print("Một tiến trình đã chết — xem /content/server.log và /content/tunnel.log")
                break
except KeyboardInterrupt:
    print("Đã dừng giữ phiên (server và tunnel VẪN chạy nền, tunnel vẫn công khai — chạy mục 12 để tắt).")

## 11. Tải kết quả về

Gói `emoca_colab_results.zip` gồm: ảnh/`.npy` của M0-04, ảnh chồng landmark của M0-05, một phản hồi mẫu, phiên bản môi trường (`env_work38.yml`, `pip_freeze_work38.txt`), số đo thời gian/VRAM (`measurements.json`) và nhật ký máy chủ. **Không** chứa token. Gửi cho nhóm để ghi vào `docs/qa/M0/` và làm fixture vàng (M1-01 AC2).

In [ ]:
import shutil
shutil.copy("/content/server.log", "/content/out/server.log")
objs = sorted(glob.glob("/content/out/**/*.obj", recursive=True))
print(f"{len(objs)} tệp .obj, tổng {sum(os.path.getsize(p) for p in objs) / 2**20:.0f} MB (từ --save_mesh)")
os.makedirs("/content/obj_skip", exist_ok=True)
for p in objs[2:]:   # giữ 2 mesh đầu làm bằng chứng M0-04; phần còn lại để ngoài gói cho tải về nhanh
    shutil.move(p, "/content/obj_skip/" + p.strip("/").replace("/", "__"))
leak = [p for p in glob.glob("/content/out/**/*", recursive=True) if os.path.isfile(p) and TOKEN.encode() in open(p, "rb").read()]
assert not leak, f"token nằm trong {leak} — không đóng gói"
shutil.make_archive("/content/emoca_colab_results", "zip", "/content/out")
from google.colab import files
print(f"Gói kết quả: {os.path.getsize('/content/emoca_colab_results.zip') / 2**20:.1f} MB")
files.download("/content/emoca_colab_results.zip")

## 12. Tắt server và tunnel

Dừng tunnel công khai và dịch vụ EMOCA khi không dùng nữa (dừng ô giữ phiên KHÔNG tắt chúng). Sau ô này địa chỉ tunnel cũ vô hiệu; muốn dùng lại thì chạy lại từ mục 7.

In [ ]:
for _name, _proc in (("tunnel", tunnel), ("server", server)):
    if _proc.poll() is None:
        _proc.terminate()
        try:
            _proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            _proc.kill()
            _proc.wait()
    print(_name, "đã dừng, mã thoát", _proc.returncode)
# `server` là `bash -c "… && python -u server.py …"`: dừng bash có thể chưa dừng tiến trình python con ⇒ dừng theo tên.
!pkill -f "python -u [s]erver.py --config /content/svc" || true  # [s]: mẫu không tự khớp dòng lệnh của chính nó
try:
    call(LOCAL, "/v1/health", TOKEN, timeout=3)
    print("CẢNH BÁO: server vẫn trả lời ở", LOCAL, "— kiểm tra bằng !ps aux | grep server.py")
except (urllib.error.URLError, ConnectionError, OSError):
    print("Đã tắt: server không còn nghe ở", LOCAL, "và tunnel đã dừng.")
